# 🌿 AbUti Spinach — AI Agricultural Assistant

**Live Demo:** [https://abutispinach.lovable.app](https://abutispinach.lovable.app)

---

## 📋 Project Overview

**AbUti Spinach** is a voice-first, AI-powered agricultural assistant designed for **smallholder farmers in South Africa**. It lives inside a glowing green orb and provides:

- 🗣️ **Voice interaction** — speak to it, it speaks back
- 📸 **Camera-based crop diagnosis** — snap a photo of a sick plant or pest
- 🌦️ **Weather-aware advice** — hyper-local weather data integrated into farming tips
- 🛒 **Shopping assistant** — where to buy seeds, fertilizer, pesticides with price comparisons
- 📱 **Offline access** — USSD and SMS fallback for areas with no internet
- 🧅 **Guided onboarding** — interactive tour that introduces features step by step

### 🎯 Problem Statement
Smallholder farmers in Africa lack access to affordable, personalized agricultural advice. Extension officers are scarce, and existing tools require literacy and internet access.

### 💡 Solution
A conversational AI assistant that works via **voice, camera, SMS, and USSD** — meeting farmers where they are, on any device.

## 🏗️ Architecture

```
┌─────────────────────────────────────────────────┐
│                  FRONTEND                        │
│         React + TypeScript + Vite                │
│                                                  │
│  ┌──────────┐ ┌──────────────┐ ┌─────────────┐  │
│  │ GreenOrb │ │OrbController │ │CameraCapture│  │
│  │(Animated │ │(Voice, Chat, │ │(Crop/Pest   │  │
│  │ Avatar)  │ │ Onboarding)  │ │ Diagnosis)  │  │
│  └──────────┘ └──────────────┘ └─────────────┘  │
│       │              │                │          │
│  SpeechSynthesis  SpeechRecognition  Camera API  │
│  (Browser TTS)    (Browser STT)     (getUserMedia)│
└───────────────────┬──────────────────────────────┘
                    │ HTTPS
┌───────────────────▼──────────────────────────────┐
│              BACKEND (Edge Functions)             │
│                                                   │
│  ┌────────────┐  ┌───────────────┐               │
│  │ /chat      │  │ /voice-callback│               │
│  │ AI Chat +  │  │ Telephony     │               │
│  │ Vision     │  │ Integration   │               │
│  └─────┬──────┘  └───────────────┘               │
│        │                                          │
│  ┌─────▼──────┐  ┌───────────────┐               │
│  │ Lovable AI │  │/ussd-callback │               │
│  │ Gateway    │  │ Offline Access │               │
│  │(Gemini/GPT)│  └───────────────┘               │
│  └────────────┘  ┌───────────────┐               │
│                  │/sms-callback  │               │
│                  │ SMS Fallback  │               │
│                  └───────────────┘               │
│                                                   │
│              Database: PostgreSQL                 │
│         (farmers, otp_codes tables)               │
└───────────────────────────────────────────────────┘
```

## 🛠️ Tech Stack

| Layer | Technology | Purpose |
|-------|-----------|----------|
| Frontend | React + TypeScript + Vite | Fast, modern SPA |
| Styling | Tailwind CSS | Utility-first responsive design |
| AI Models | Google Gemini 3 Flash / Gemini 2.5 Pro | Chat + Vision (crop analysis) |
| Voice Input | Web Speech API (SpeechRecognition) | Browser-native STT |
| Voice Output | Web Speech API (SpeechSynthesis) | Browser-native TTS |
| Backend | Supabase Edge Functions (Deno) | Serverless API endpoints |
| Database | PostgreSQL (Supabase) | Farmer profiles, OTP codes |
| Weather | Open-Meteo API | Free weather data, no API key |
| Telephony | Africa's Talking | USSD, SMS, Voice callbacks |
| Hosting | Lovable Cloud | Auto-deployed, CDN-backed |

## 🧠 Core AI: Chat Edge Function

The brain of AbUti Spinach is a serverless edge function that routes messages to AI models via the Lovable AI Gateway. It supports both **text chat** and **vision** (image analysis for crop/pest diagnosis).

### Key Design Decisions:
- **Model switching**: Uses `gemini-3-flash-preview` for text, upgrades to `gemini-2.5-pro` when images are attached
- **Streaming responses**: SSE streaming for real-time typing effect
- **Character persona**: "AbUti Spinach" — sarcastic but genuinely helpful farming expert
- **Shopping assistant**: Recommends where to buy supplies with price comparisons by region

In [ ]:
# Edge Function: /chat (Deno/TypeScript)
# This is the actual production code running on the backend

chat_function_code = '''
serve(async (req) => {
  const { messages } = await req.json();
  
  // Smart model selection: vision model for images, fast model for text
  const hasImage = messages.some((m) => m.image);
  const model = hasImage ? "google/gemini-2.5-pro" : "google/gemini-3-flash-preview";

  // Transform messages to support multimodal content (text + images)
  const transformedMessages = messages.map((m) => {
    if (m.image) {
      return {
        role: m.role,
        content: [
          { type: "text", text: m.content },
          { type: "image_url", image_url: { url: `data:image/jpeg;base64,${m.image}` } },
        ],
      };
    }
    return m;
  });

  // Stream response from AI gateway
  const response = await fetch(AI_GATEWAY_URL, {
    method: "POST",
    body: JSON.stringify({ model, messages: [systemPrompt, ...transformedMessages], stream: true }),
  });

  return new Response(response.body, { headers: { "Content-Type": "text/event-stream" } });
});
'''

print('Chat edge function handles both text and image-based queries')
print('Streaming SSE responses for real-time UX')
print('Automatic model upgrade for vision tasks')

## 🌿 The Green Orb — Animated AI Avatar

The orb is a **fully animated character** rendered with pure CSS/JS (no external animation libraries for the face). It has:

- **12 emotional states**: neutral, curious, happy, laughing, sleepy, sleeping, thinking, speaking, listening, worried, excited, eyebrow-raise
- **Eye tracking**: Eyes follow mouse/touch position
- **Micro-expressions**: Random eyebrow raises, smirks, squints
- **Mouth animation**: Synced to speech output with multiple mouth frames
- **Idle behavior**: Falls asleep after inactivity
- **Dynamic aura**: Glow color/intensity changes with emotion

In [ ]:
# Emotion system
emotions = [
    'neutral',       # Default resting state
    'curious',       # Raised eyebrow, wider eyes
    'happy',         # Smile, squinted eyes
    'laughing',      # Big grin, shaking
    'sleepy',        # Droopy eyes
    'sleeping',      # Closed eyes, zzz
    'thinking',      # Eyes look up, mouth pursed
    'speaking',      # Mouth animates, sound waves
    'listening',     # Ears perk up, radar sweep
    'worried',       # Furrowed brow
    'excited',       # Wide eyes, bouncing
    'eyebrow-raise'  # Sassy reaction
]

print(f'{len(emotions)} distinct emotional states')
print('Eye tracking follows cursor/touch')
print('Auto-sleeps after 15s of inactivity')
print('Haptic feedback on mobile (vibration API)')

## 🗣️ Voice Interaction System

AbUti Spinach is **voice-first** — designed for farmers who may have limited literacy.

### Speech-to-Text (Input)
- Uses the **Web Speech API** (`SpeechRecognition`)
- Supports continuous listening with interim results
- Language: `en-ZA` (South African English)
- Camera triggers: phrases like "take a picture", "scan", "check my crop" auto-open the camera

### Text-to-Speech (Output)
- Uses browser-native `SpeechSynthesis`
- Prefers South African English voices when available
- Cleans markdown/emoji from AI responses before speaking
- Synced with orb mouth animation

In [ ]:
# Voice interaction flow
voice_flow = {
    '1. User taps orb': 'Orb switches to listening emotion, radar sweep animation',
    '2. SpeechRecognition starts': 'Browser captures audio, converts to text',
    '3. Text sent to /chat': 'Edge function streams AI response via SSE',
    '4. Response displayed': 'Caption bubble shows text with typing effect',
    '5. TTS speaks response': 'Browser reads response aloud, orb mouth animates',
    '6. Cycle repeats': 'Orb returns to listening or idle state',
}

for step, desc in voice_flow.items():
    print(f'{step}: {desc}')

# Camera trigger keywords
camera_triggers = [
    'take a picture', 'take photo', 'scan', 'camera',
    'show me', 'look at', 'analyze this', 'what is this',
    'pest', 'disease', 'check my crop'
]
print(f'\n{len(camera_triggers)} voice commands trigger the camera')

## 📸 Camera-Based Crop Diagnosis

Farmers can photograph sick plants, pests, or soil issues. The image is:

1. Captured via `getUserMedia` (device camera)
2. Compressed and converted to base64
3. Sent to the `/chat` endpoint alongside a text query
4. Processed by **Gemini 2.5 Pro** (vision model)
5. AI identifies the issue and recommends treatment + where to buy it

### Example Diagnosis Flow:
```
Farmer: "What's wrong with my tomato plant?" [+ photo]
AbUti:  "That's early blight (Alternaria solani)
         Those dark spots with concentric rings are textbook.
         Get Mancozeb fungicide - Agrimark has it for R65.
         Remove affected leaves NOW before it spreads."
```

## 🌦️ Weather Integration

Uses the **Open-Meteo API** (completely free, no API key required) to provide:

- Current temperature, humidity, wind speed
- Daily max/min temperature
- Precipitation sum and probability
- Weather description (clear, cloudy, rain, etc.)
- Reverse geocoding for location name

Weather data is injected into the AI's first message context, enabling proactive advice like:
> *"It's 34C with 15% humidity in Pretoria — your spinach is basically sunbathing without sunscreen. Water deeply this evening."*

In [ ]:
# Weather data structure
weather_data_example = {
    'temperature': 28.5,
    'humidity': 45,
    'windSpeed': 12.3,
    'description': 'Partly cloudy',
    'daily': {
        'maxTemp': 32,
        'minTemp': 18,
        'precipitationSum': 0,
        'precipitationProbability': 10
    },
    'locationName': 'Pretoria, South Africa',
    'latitude': -25.7479,
    'longitude': 28.2293
}

print('Weather data example:')
for key, value in weather_data_example.items():
    print(f'  {key}: {value}')

print('\nFree API - no key required (Open-Meteo)')
print('Location via browser Geolocation API')
print('Reverse geocoded to human-readable place name')

## 📱 Offline Access: USSD and SMS

For farmers with feature phones or no internet, AbUti Spinach is accessible via:

### USSD (`*384*87343#`)
- Menu-driven interface on any phone
- No internet required
- Powered by Africa's Talking API
- Edge function: `/ussd-callback`

### SMS
- Send a text message with your farming question
- Get an AI-powered response back via SMS
- Edge function: `/sms-callback`

### Voice Call
- Call in and speak your question
- Edge function: `/voice-callback`

This ensures **digital inclusion** — the solution works for the 60%+ of African farmers without smartphones.

In [ ]:
# USSD callback edge function (simplified)
ussd_code = '''
serve(async (req) => {
  const formData = await req.formData();
  const sessionId = formData.get("sessionId");
  const text = formData.get("text") || "";
  const parts = text.split("*").filter(Boolean);

  if (text === "") {
    // Main menu
    return "CON Welcome to abuti Spinach\n"
         + "1. Ask a farming question\n"
         + "2. Get weather advice\n"
         + "3. Crop recommendations\n"
         + "4. Pest & disease help";
  }

  // Route to AI based on menu choice
  const aiResponse = await getAIResponse(userQuery);
  return `END abuti Spinach:\n${truncate(aiResponse, 160)}`;
});
'''

print('USSD Menu Structure:')
print('  *384*87343# -> Main Menu')
print('  Option 1 -> Free-text farming question')
print('  Option 2 -> Weather advice by region')
print('  Option 3 -> Crop recs by soil type')
print('  Option 4 -> Pest/disease diagnosis')
print('\nAll responses truncated to 160 chars for USSD')

## 🧪 Live Test Results

These tests were run against the **live deployed edge functions**:

### USSD Tests

| Test | Input | Result |
|------|-------|--------|
| Main menu | `text=""` | CON Welcome to abuti Spinach... (4 options) |
| Option 1 | `text="1"` | CON Type your farming question: |
| Option 3 | `text="3"` | CON What's your soil type? (5 options) |
| Full AI query | `text="1*How do I grow tomatoes"` | END abuti Spinach: Plant in sun with compost... |

All tests returned HTTP 200 with correct USSD `CON`/`END` prefixes.

## 🎭 Onboarding Flow

New users get a guided interactive tour:

1. **Greeting** — AbUti introduces itself with sass
2. **Name collection** — Voice or text input
3. **Location permission** — For weather-based advice
4. **Feature tour** — Voice, Camera, USSD/SMS explained
5. **Ready to chat** — Personalized with farmer's name

Returning users skip the tour and go straight to chat. User data persisted in localStorage + database.

## 🗄️ Database Schema

In [ ]:
# Database schema
schema = {
    'farmers': {
        'id': 'UUID (primary key)',
        'name': 'TEXT (farmer name)',
        'phone': 'TEXT (phone number)',
        'created_at': 'TIMESTAMP',
        'last_login': 'TIMESTAMP (nullable)'
    },
    'otp_codes': {
        'id': 'UUID (primary key)',
        'phone': 'TEXT',
        'code': 'TEXT (6-digit OTP)',
        'used': 'BOOLEAN (default false)',
        'created_at': 'TIMESTAMP',
        'expires_at': 'TIMESTAMP'
    }
}

for table, columns in schema.items():
    print(f'\nTable: {table}')
    for col, desc in columns.items():
        print(f'   {col}: {desc}')

## 🔐 Security

- **Row Level Security (RLS)** enabled on all tables
- **JWT verification disabled** on public-facing edge functions (USSD/SMS callbacks need to be publicly accessible)
- **API keys stored as secrets** — never in source code
- **CORS headers** configured for cross-origin requests
- **No user auth required** for basic chat — reducing friction for first-time farmers

## 🚀 How to Run / Test

### Option 1: Live Demo (Recommended)
Open on your phone for the best experience (voice + camera):

**https://abutispinach.lovable.app**

### Option 2: Clone and Run Locally
```bash
git clone <repo-url>
cd abuti-spinach
npm install
npm run dev
```

### Option 3: USSD/SMS (Feature Phone)
- Dial `*384*87343#` on any phone
- Or send an SMS to the configured number

### What to Test:
1. **Tap the green orb** to start onboarding
2. **Say your name** — it remembers you
3. **Ask a farming question** — "How do I grow tomatoes?"
4. **Say "take a picture"** — camera opens for crop diagnosis
5. **Ask about weather** — gives location-based advice
6. **Ask where to buy something** — "Where can I buy NPK fertilizer?"

## 📈 Impact and Scalability

| Metric | Value |
|--------|-------|
| Target users | 33M+ smallholder farmers in Southern Africa |
| Access channels | Web, Voice, Camera, USSD, SMS |
| Languages | English (ZA), expandable to Zulu, Sotho, Xhosa |
| Cost per query | ~$0.001 (Gemini Flash) |
| Offline capable | Yes (USSD/SMS) |
| API keys needed | 0 for core features (weather + AI via Lovable) |

### Future Roadmap
- Multi-language support (isiZulu, Sesotho, isiXhosa)
- Crop calendar and planting reminders
- Marketplace: connect farmers to buyers
- Satellite imagery for field health monitoring
- Micro-insurance integration

---

## Thank You

**AbUti Spinach** — *Because every farmer deserves a sarcastic genius in their pocket.*

**Live Demo:** [https://abutispinach.lovable.app](https://abutispinach.lovable.app)

---
*Built with Lovable, React, Gemini AI, and a questionable amount of farming puns.*